# 07 — Auditar e localizar CEPs pelo CNEFE

**Ideia:** o CNEFE (IBGE) tem, para cada endereço, o **CEP** e a **coordenada**. Os endereços de um CEP formam uma nuvem de pontos que mostra **onde o CEP está de fato**. Este notebook usa isso para três coisas, sobre a **base oficial inteira** (`02_dados_editados\logradouros_cep.gpkg`):

| Parte | Pergunta | Como |
|---|---|---|
| **A – Auditoria** | Os CEPs já associados estão no lugar certo? | Distância de cada ponto do CEP até os segmentos onde o CEP está na base. Mediana alta = **suspeito** |
| **B – Classe C** (via ambígua) | Qual dos CEPs candidatos vai em cada trecho? | Votação: pontos do CNEFE mais próximos do trecho cujo CEP está entre os candidatos |
| **C – Classe D** (CEP órfão) | Em que trecho fica um CEP dos Correios que não está na base? | Votação **por segmento**: cada segmento recebe o CEP órfão com mais pontos ligados a ele (sugestão **por localização**, não por nome) |

**Conceitos:**
- **local** = coordenada distinta do CNEFE (um prédio com 80 apartamentos conta 1 voto, não 80);
- o ponto é ligado ao **segmento mais próximo** da base (até `dist_busca_m`);
- **participação** = votos do vencedor ÷ votos válidos. Só vira `sugerido` com pelo menos `min_pontos` locais e participação ≥ `min_participacao`.

**Nada aqui altera a base.** Você revisa no QGIS (camada `sugestoes_segmentos`, ligada pelo `id`) e copia para a coluna `cep` o que aceitar, com `cep_origem = manual`.

**Limites:** o CEP do CNEFE vem do Censo 2022 e pode ter erros pontuais; trechos sem edificação (rodovias, áreas vazias) não têm pontos e continuam manuais.

**Saídas** (`03_processamento\07_auditar_cep_cnefe`): `auditoria_cnefe.gpkg` (camadas `sugestoes_segmentos`, `auditoria_segmentos`, `pontos_cnefe`) e `auditoria_cnefe.xlsx`.

**Estrutura:** 1 Preparação · 2 Base oficial e Correios · 3 CNEFE · 4 Ligar pontos aos segmentos · 5 A – Auditoria · 6 B – Classe C · 7 C – Classe D · 8 Sugestões por segmento · 9 Exportação · 10 Resumo

## 1. Preparação e configuração
Parâmetros na seção `parametros.cnefe` do `config.yaml`.

In [ ]:
# --- Preparação padrão: acha a raiz do projeto, lê o config.yaml e liga o log (tela + arquivo) -------------
import sys                                            # para o Python achar as funções comuns (scripts/src)
from pathlib import Path                              # caminhos de arquivos
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())   # sobe pastas até achar o config.yaml
sys.path.insert(0, str(RAIZ / "scripts"))             # libera o "from src import ..."
from src import config, registro                      # funções comuns do padrão de projetos
cfg = config.carregar(RAIZ)                           # lê o config.yaml (caminhos e parâmetros)
ETAPA = "07_auditar_cep_cnefe"              # nome desta etapa (= nome do notebook)
log = registro.iniciar_log(ETAPA, cfg)                # log na tela e em 03_processamento/logs/
PASTA_SAIDA = cfg.pasta_etapa(ETAPA)                  # 03_processamento/<ETAPA> (criada se não existir)
import numpy as np                    # contas com colunas
import pandas as pd                   # tabelas
import pyogrio                        # leitura de arquivos geográficos
import geopandas as gpd               # tabelas com geometria
from rapidfuzz import fuzz            # similaridade de textos
from src import fontes                                   # arquivos editados (prioridade editado -> bruto)
from src.validacao import validar_tabela                 # bloco padrão de validação
from src.texto import normalizar_nome, tirar_tipo        # normalização de nomes de logradouro
from src.geo import para_gpkg, checar_crs                # tipos compatíveis com GeoPackage e CRS
from src.planilha import formatar_excel                  # formatação das abas do Excel

# --- Entradas (config.yaml) ------------------------------------------------------------------------------
ARQ_BASE = fontes.caminho_editado(cfg, "logradouros_cep")      # base oficial editada
CAMADA_BASE = cfg.camada("editados.logradouros_cep")
ARQ_CNEFE = cfg.caminho("brutos.cnefe")                        # cnefe do município
ARQ_PONTE = cfg.caminho("processamento.ponte")                 # universo de CEPs dos correios (etapa 02)
ARQ_ORFAOS = cfg.caminho("processamento.orfaos")
ARQ_DIAG = cfg.caminho("processamento.diagnostico")            # vínculos da rais por CEP (etapa 05); opcional

# --- Saídas -------------------------------------------------------------------------------------------------------
ARQ_GPKG = PASTA_SAIDA / "auditoria_cnefe.gpkg"
ARQ_EXCEL = PASTA_SAIDA / "auditoria_cnefe.xlsx"

# --- Parâmetros ------------------------------------------------------------------------------------------------------
CRS_METRICO = cfg.projeto("crs_metrico")                       # geosanja (metros)
CRS_GRAUS = cfg.projeto("crs_graus")                           # latitude/longitude do cnefe
DIST_PERTO_M = cfg.parametro("cnefe.dist_perto_m")             # ponto a até isso da linha = "na linha"
DIST_BUSCA_M = cfg.parametro("cnefe.dist_busca_m")             # distância máxima ponto -> segmento mais próximo
MEDIANA_SUSPEITA_M = cfg.parametro("cnefe.mediana_suspeita_m")   # mediana acima disso = CEP suspeito
MIN_PONTOS = cfg.parametro("cnefe.min_pontos")                 # mínimo de locais para decidir
MIN_PARTICIPACAO = cfg.parametro("cnefe.min_participacao")     # participação mínima do vencedor
COLUNAS_CNEFE = ["CEP", "NOM_TIPO_SEGLOGR", "NOM_TITULO_SEGLOGR", "NOM_SEGLOGR", "LATITUDE", "LONGITUDE"]

log.info("Configuração carregada!")
for _nome, _arq in [("Base oficial", ARQ_BASE), ("cnefe", ARQ_CNEFE), ("Ponte (correios)", ARQ_PONTE),
                    ("Órfãos (correios)", ARQ_ORFAOS), ("Diagnóstico rais (vínculos)", ARQ_DIAG)]:
    log.info("%s: %s | existe: %s", _nome, _arq.name, _arq.exists())
log.info("Limites: perto %s m | busca %s m | mediana suspeita %s m | mín. %s locais | participação %.0f%%",
         DIST_PERTO_M, DIST_BUSCA_M, MEDIANA_SUSPEITA_M, MIN_PONTOS, MIN_PARTICIPACAO * 100)

## 2. Base oficial, Correios e vínculos
- **Base:** segmentos com geometria; a coluna `cep` é separada em `|` (um par segmento–CEP por linha);
- **Correios:** universo de CEPs (etapa 02). **Órfãos** = CEPs dos Correios que não estão em nenhum segmento da base;
- **Vínculos da RAIS** (etapa 05, se existir): só para ordenar as listas por importância.

In [ ]:
log.info("Carregando a base oficial...")
base = pyogrio.read_dataframe(ARQ_BASE, layer=CAMADA_BASE,
                              columns=["id", "tipo", "nome", "loteamento", "status_cep", "cep", "cep_origem", "ceps_candidatos"])
checar_crs(base, CRS_METRICO, "base oficial")
for col in ["tipo", "nome", "loteamento", "status_cep", "cep", "cep_origem", "ceps_candidatos"]:
    base[col] = base[col].astype(object).where(base[col].notna(), "").astype(str).str.strip()
base["via"] = (base["tipo"] + " " + base["nome"]).str.strip()
validar_tabela("Base oficial", base, chave=["id"], obrigatorias=["id"])
info_seg = pd.DataFrame(base[["id", "via", "loteamento", "cep", "cep_origem"]]).rename(columns={"id": "id_seg"})   # atributos por segmento

# --- Um par (segmento, CEP) por linha ------------------------------------------------------------------------------
pares = base.loc[base["cep"] != "", ["id", "cep", "cep_origem", "via"]].copy()
pares["cep"] = pares["cep"].str.split("|")
pares = pd.DataFrame(pares.explode("cep"))
pares["cep"] = pares["cep"].str.replace(r"\D", "", regex=True)
pares = pares[pares["cep"].str.len() == 8].drop_duplicates(["id", "cep"]).reset_index(drop=True)
ceps_base = set(pares["cep"])
log.info("Pares segmento-CEP: %d | CEPs distintos na base: %d", len(pares), len(ceps_base))

log.info("Carregando o universo dos correios...")
ponte = pd.read_parquet(ARQ_PONTE, columns=["cep", "logradouro_correios", "bairro_correios"])
orfaos_e1 = pd.read_csv(ARQ_ORFAOS, sep=";", dtype=str, encoding="utf-8-sig", keep_default_na=False)
universo = pd.concat([
    ponte.groupby("cep", as_index=False).agg(logradouro=("logradouro_correios", "first"), bairro=("bairro_correios", "first")),
    orfaos_e1.rename(columns={"logradouro_orig": "logradouro", "bairro_orig": "bairro"})[["cep", "logradouro", "bairro"]]
]).drop_duplicates("cep").reset_index(drop=True)
orfaos = universo[~universo["cep"].isin(ceps_base)].copy()
log.info("CEPs nos correios: %d | órfãos (fora da base): %d", len(universo), len(orfaos))

if ARQ_DIAG.exists():
    vinc = pd.read_parquet(ARQ_DIAG, columns=["cep", "classe", "vinc_ativos_ref", "vinc_ativos_soma_anos"])
    log.info("Vínculos da rais carregados para %d CEPs.", len(vinc))
else:
    vinc = pd.DataFrame({"cep": pd.Series(dtype=str), "classe": pd.Series(dtype=str),
                         "vinc_ativos_ref": pd.Series(dtype=float), "vinc_ativos_soma_anos": pd.Series(dtype=float)})
    log.warning("Diagnóstico da etapa 05 não encontrado: as listas não terão vínculos da rais.")
log.info("Base e correios carregados!")

## 3. CNEFE → locais com CEP
Lê só as colunas necessárias, mantém endereços com CEP de 8 dígitos e coordenada, e junta os endereços de mesma coordenada e mesmo CEP num **local** (`n_enderecos` guarda quantos eram). Converte de graus para metros (`crs_metrico`).

In [ ]:
log.info("Carregando o cnefe...")
def ler_cnefe(codificacao):
    """Lê o cnefe (separador ';') só com as colunas necessárias, tudo como texto."""
    return pd.read_csv(ARQ_CNEFE, sep=";", dtype=str, usecols=COLUNAS_CNEFE, encoding=codificacao, keep_default_na=False)

try:
    cnefe = ler_cnefe("utf-8")
except UnicodeDecodeError:
    log.warning("Arquivo não está em UTF-8; relendo como Latin-1.")
    cnefe = ler_cnefe("latin-1")
n_cnefe = len(cnefe)
cnefe["cep"] = cnefe["CEP"].str.replace(r"\D", "", regex=True)
cnefe["lat"] = pd.to_numeric(cnefe["LATITUDE"].str.replace(",", ".", regex=False), errors="coerce")
cnefe["lon"] = pd.to_numeric(cnefe["LONGITUDE"].str.replace(",", ".", regex=False), errors="coerce")
cnefe["rua_cnefe"] = (cnefe["NOM_TIPO_SEGLOGR"] + " " + cnefe["NOM_TITULO_SEGLOGR"] + " " + cnefe["NOM_SEGLOGR"]).str.split().str.join(" ")
validos = (cnefe["cep"].str.len() == 8) & cnefe["lat"].notna() & cnefe["lon"].notna()
log.info("Endereços no cnefe: %d | com CEP e coordenada: %d (%.1f%%)", n_cnefe, int(validos.sum()), validos.mean() * 100)

# --- Locais: endereços com mesma coordenada e mesmo CEP viram um só ------------------------------------------------
locais = (cnefe[validos].groupby(["cep", "lat", "lon"], as_index=False)
          .agg(n_enderecos=("cep", "size"), rua_cnefe=("rua_cnefe", "first")))
locais["loc_id"] = np.arange(len(locais), dtype="int64")
locais = gpd.GeoDataFrame(locais, geometry=gpd.points_from_xy(locais["lon"], locais["lat"]), crs=CRS_GRAUS).to_crs(CRS_METRICO)
log.info("Locais (coordenadas distintas por CEP): %d | CEPs distintos no cnefe: %d", len(locais), locais["cep"].nunique())
validar_tabela("Locais do cnefe", locais, chave=["loc_id"], obrigatorias=["cep", "loc_id"])
log.info("cnefe carregado!")

## 4. Ligar cada local ao segmento mais próximo
Cada local é ligado ao segmento **mais próximo** da base inteira (até `dist_busca_m`). Essa ligação é usada nas partes B e C e para mostrar onde estão os pontos dos CEPs suspeitos.

In [ ]:
log.info("Ligando os locais ao segmento mais próximo (até %s m)...", DIST_BUSCA_M)
segs = base[["id", "geometry"]].rename(columns={"id": "id_seg"})
lig = gpd.sjoin_nearest(locais[["loc_id", "geometry"]], segs, how="left", max_distance=DIST_BUSCA_M, distance_col="dist_seg")
lig = pd.DataFrame(lig[["loc_id", "id_seg", "dist_seg"]]).drop_duplicates("loc_id")      # empate de distância: fica um
locais = locais.merge(lig, on="loc_id", how="left")
if pd.api.types.is_integer_dtype(base["id"]):
    locais["id_seg"] = locais["id_seg"].astype("Int64")                                  # mesmo tipo do id da base
com_seg = int(locais["id_seg"].notna().sum())
log.info("Locais com segmento a até %s m: %d de %d (%.1f%%)", DIST_BUSCA_M, com_seg, len(locais), com_seg / len(locais) * 100 if len(locais) else 0)
assert locais["loc_id"].is_unique, "A ligação duplicou locais!"


def fmt_ids(valores, maximo=15):
    """Lista de ids em texto 'a|b|c' (no máximo 'maximo' ids, depois '...')."""
    v = sorted({int(x) if isinstance(x, (int, np.integer, float, np.floating)) and float(x).is_integer() else x for x in valores})
    return "|".join(str(x) for x in v[:maximo]) + ("|..." if len(v) > maximo else "")


def onde_estao(loc):
    """Para cada CEP, a via (tipo + nome) que concentra os locais ligados a segmentos.

    Devolve: cep, via_top, ids_top (segmentos dessa via que receberam pontos), n_locais_top,
    n_locais_com_seg e participacao (n_locais_top / n_locais_com_seg).
    """
    x = pd.DataFrame(loc.loc[loc["id_seg"].notna(), ["cep", "loc_id", "id_seg"]]).merge(info_seg[["id_seg", "via"]], on="id_seg")
    if x.empty:
        return pd.DataFrame(columns=["cep", "via_top", "ids_top", "n_locais_top", "n_locais_com_seg", "participacao"])
    cont = x.groupby(["cep", "via"], as_index=False).agg(n_locais=("loc_id", "nunique"), ids=("id_seg", fmt_ids))
    cont["n_locais_com_seg"] = cont.groupby("cep")["n_locais"].transform("sum")
    top = cont.sort_values(["cep", "n_locais"], ascending=[True, False]).drop_duplicates("cep")
    top["participacao"] = (top["n_locais"] / top["n_locais_com_seg"]).round(3)
    return top.rename(columns={"via": "via_top", "ids": "ids_top", "n_locais": "n_locais_top"})[
        ["cep", "via_top", "ids_top", "n_locais_top", "n_locais_com_seg", "participacao"]]


log.info("Ligação concluída!")

## 5. A — Auditoria dos CEPs já associados
Para cada local de um CEP que **já está na base**, mede a distância até o segmento mais próximo **entre os segmentos daquele CEP**. Por CEP:
- `dist_mediana_m`, `dist_max_m`, `pct_perto` (% de locais a até `dist_perto_m`);
- **status:** `sem_pontos_cnefe` · `poucos_pontos` (< `min_pontos`) · `suspeito` (mediana > `mediana_suspeita_m`) · `atencao` (menos da metade dos locais perto da linha) · `ok`;
- para suspeitos e atenção: **onde os pontos estão** (via e segmentos que concentram os locais).

In [ ]:
log.info("Auditando os CEPs já associados...")
geom_seg = dict(zip(base["id"], base.geometry))                                # id -> geometria do segmento
pa = locais.loc[locais["cep"].isin(ceps_base), ["loc_id", "cep", "geometry"]]
m = pa.merge(pares[["cep", "id"]], on="cep")                                   # cada local x cada segmento do seu CEP
linhas_cep = gpd.GeoSeries([geom_seg[i] for i in m["id"]], index=m.index, crs=CRS_METRICO)
m["dist_cep"] = m.geometry.distance(linhas_cep)                                # distância ponto -> segmento (metros)
dist_loc = pd.DataFrame(m).groupby("loc_id", as_index=False).agg(cep=("cep", "first"), dist_cep=("dist_cep", "min"))
log.info("Locais auditados: %d (de CEPs que estão na base)", len(dist_loc))

stats = (dist_loc.groupby("cep", as_index=False)
         .agg(n_locais=("loc_id", "size"), dist_mediana_m=("dist_cep", "median"), dist_max_m=("dist_cep", "max"),
              pct_perto=("dist_cep", lambda s: (s <= DIST_PERTO_M).mean() * 100)))
por_cep_base = (pares.groupby("cep", as_index=False)
                .agg(n_segmentos=("id", "nunique"), ids_base=("id", fmt_ids), vias_base=("via", lambda s: " ; ".join(sorted(set(s)))[:200]),
                     origem=("cep_origem", lambda s: "|".join(sorted(set(s))))))
aud = por_cep_base.merge(stats, on="cep", how="left")
n = aud["n_locais"].fillna(0)
aud["status"] = np.select(
    [n == 0, n < MIN_PONTOS, aud["dist_mediana_m"] > MEDIANA_SUSPEITA_M, aud["pct_perto"] < 50],
    ["sem_pontos_cnefe", "poucos_pontos", "suspeito", "atencao"], default="ok")

# --- Onde estão os pontos dos CEPs suspeitos/atenção (só os locais longe da linha do CEP) ---------------------------------
alerta = set(aud.loc[aud["status"].isin(["suspeito", "atencao"]), "cep"])
longe = dist_loc.loc[dist_loc["cep"].isin(alerta) & (dist_loc["dist_cep"] > DIST_PERTO_M), "loc_id"]
onde = onde_estao(locais[locais["loc_id"].isin(longe)]).rename(columns={
    "via_top": "via_dos_pontos", "ids_top": "ids_dos_pontos", "n_locais_top": "n_locais_via_pontos",
    "n_locais_com_seg": "n_locais_longe_com_seg", "participacao": "participacao_via_pontos"})
aud = aud.merge(onde, on="cep", how="left").merge(vinc, on="cep", how="left")
for col in ["dist_mediana_m", "dist_max_m", "pct_perto"]:
    aud[col] = aud[col].round(1)
ordem = {"suspeito": 0, "atencao": 1, "poucos_pontos": 2, "sem_pontos_cnefe": 3, "ok": 4}
aud = aud.sort_values(["status", "vinc_ativos_soma_anos"], key=lambda s: s.map(ordem) if s.name == "status" else -s.fillna(0)).reset_index(drop=True)
log.info("Auditoria por status:\n%s", aud["status"].value_counts().reindex(list(ordem), fill_value=0).to_string())
validar_tabela("Auditoria por CEP", aud, chave=["cep"], obrigatorias=["cep", "status"], linhas_entrada=len(ceps_base))
log.info("Auditoria concluída!")

## 6. B — Classe C: CEP de cada trecho ambíguo
Trecho ambíguo = segmento **sem** `cep` e **com** `ceps_candidatos`. Votam os locais ligados ao trecho (segmento mais próximo) cujo CEP está entre os candidatos. Também conta quantos locais ligados ao trecho têm **outro** CEP (informativo).

In [ ]:
log.info("Votando o CEP dos trechos ambíguos (classe C)...")
amb = base.loc[(base["cep"] == "") & (base["ceps_candidatos"] != ""), ["id", "via", "loteamento", "ceps_candidatos"]].copy()
amb_long = amb[["id", "ceps_candidatos"]].assign(cep=amb["ceps_candidatos"].str.split("|")).explode("cep")
amb_long["cep"] = amb_long["cep"].str.replace(r"\D", "", regex=True)
amb_long = pd.DataFrame(amb_long[amb_long["cep"].str.len() == 8])
log.info("Trechos ambíguos: %d", len(amb))

liga_amb = pd.DataFrame(locais.loc[locais["id_seg"].isin(amb["id"]), ["loc_id", "cep", "id_seg"]])
votos = liga_amb.merge(amb_long[["id", "cep"]], left_on=["id_seg", "cep"], right_on=["id", "cep"])
cont = votos.groupby(["id", "cep"], as_index=False).agg(n_locais=("loc_id", "nunique"))
cont["n_votos_validos"] = cont.groupby("id")["n_locais"].transform("sum")
cont = cont.sort_values(["id", "n_locais"], ascending=[True, False])
detalhe = cont.groupby("id")[["cep", "n_locais"]].apply(lambda d: " | ".join(f"{c}: {k}" for c, k in zip(d["cep"], d["n_locais"]))).rename("votos")
top = cont.drop_duplicates("id").rename(columns={"cep": "cep_sugerido", "n_locais": "n_locais_vencedor"})
top["participacao"] = (top["n_locais_vencedor"] / top["n_votos_validos"]).round(3)
outros = (liga_amb.merge(amb_long[["id", "cep"]].assign(cand=True), left_on=["id_seg", "cep"], right_on=["id", "cep"], how="left")
          .query("cand != True").groupby("id_seg").size().rename("n_locais_outros_ceps"))

sug_c = (amb.merge(top, on="id", how="left").merge(detalhe, left_on="id", right_index=True, how="left")
            .merge(outros, left_on="id", right_index=True, how="left"))
sug_c["n_locais_outros_ceps"] = sug_c["n_locais_outros_ceps"].fillna(0).astype(int)
nv = sug_c["n_locais_vencedor"].fillna(0)
sug_c["status"] = np.select([nv == 0, (nv >= MIN_PONTOS) & (sug_c["participacao"] >= MIN_PARTICIPACAO)],
                            ["sem_pontos", "sugerido"], default="votacao_fraca")
sug_c = sug_c.merge(vinc[["cep", "vinc_ativos_soma_anos"]].rename(columns={"cep": "cep_sugerido"}), on="cep_sugerido", how="left")
log.info("Trechos ambíguos por status:\n%s", sug_c["status"].value_counts().to_string())
validar_tabela("Sugestões classe C", sug_c, chave=["id"], obrigatorias=["id", "status"], linhas_entrada=len(amb))
log.info("Classe C concluída!")

## 7. C — Classe D: trechos de cada CEP órfão
CEP órfão = CEP dos Correios que **não está** em nenhum segmento da base. A decisão é **por segmento**: cada segmento (fora dos trechos ambíguos, decididos na parte B) recebe o CEP órfão que tem **mais locais ligados a ele**. Assim, uma rua com trechos de CEPs diferentes recebe um CEP por trecho.

Por CEP, a tabela mostra os segmentos que ele **venceu** (`ids_sugeridos`), a via onde está a maioria dos seus pontos (`via_top`) e a **concordância de nome** (`score_nome`, 0–100) entre o logradouro dos Correios e essa via.

**status (por CEP):** `sugerido` · `votacao_fraca` · `em_trecho_ambiguo` (venceu um trecho na parte B) · `sem_trecho_vencido` (tem pontos perto de segmentos, mas não é maioria em nenhum) · `pontos_longe` (nenhum local a até `dist_busca_m` de um segmento) · `sem_pontos_cnefe`.

In [ ]:
log.info("Localizando os CEPs órfãos (classe D)...")
set_orf = set(orfaos["cep"])
loc_d = locais[locais["cep"].isin(set_orf)]
por_cep = (pd.DataFrame(loc_d).groupby("cep", as_index=False)
           .agg(n_locais=("loc_id", "size"), n_enderecos=("n_enderecos", "sum"),
                rua_cnefe=("rua_cnefe", lambda s: s.value_counts().index[0] if len(s) else "")))

# --- Votação por segmento (os trechos ambíguos ficam com a parte B) --------------------------------------------------------
vd = pd.DataFrame(loc_d.loc[loc_d["id_seg"].notna() & ~loc_d["id_seg"].isin(amb["id"]), ["loc_id", "cep", "id_seg"]])
cont_d = vd.groupby(["id_seg", "cep"], as_index=False).agg(n_locais=("loc_id", "nunique"))
cont_d["n_votos_validos"] = cont_d.groupby("id_seg")["n_locais"].transform("sum")
cont_d = cont_d.sort_values(["id_seg", "n_locais"], ascending=[True, False])
votos_d = (cont_d.groupby("id_seg")[["cep", "n_locais"]]
           .apply(lambda d: " | ".join(f"{c}: {k}" for c, k in zip(d["cep"], d["n_locais"]))).rename("votos"))
seg_d = cont_d.drop_duplicates("id_seg").rename(columns={"cep": "cep_sugerido", "n_locais": "n_locais_vencedor"})
seg_d["participacao"] = (seg_d["n_locais_vencedor"] / seg_d["n_votos_validos"]).round(3)
seg_d["status"] = np.where((seg_d["n_locais_vencedor"] >= MIN_PONTOS) & (seg_d["participacao"] >= MIN_PARTICIPACAO),
                           "sugerido", "votacao_fraca")
seg_d = seg_d.merge(votos_d, left_on="id_seg", right_index=True, how="left")
log.info("Segmentos (fora dos ambíguos) com voto de CEP órfão: %d | sugerido: %d", len(seg_d), int((seg_d["status"] == "sugerido").sum()))

# --- Por CEP: segmentos que ele venceu --------------------------------------------------------------------------------------
ganhos = (seg_d.groupby("cep_sugerido", as_index=False)
          .agg(ids_sugeridos=("id_seg", fmt_ids), n_segmentos_sugeridos=("id_seg", "size"),
               melhor_status=("status", lambda s: "sugerido" if (s == "sugerido").any() else "votacao_fraca"))
          .rename(columns={"cep_sugerido": "cep"}))
sug_d = (orfaos.merge(por_cep, on="cep", how="left").merge(onde_estao(loc_d), on="cep", how="left")
               .merge(ganhos, on="cep", how="left"))

def score_nome(a, b):
    """Similaridade (0-100) entre dois nomes de logradouro, sem o tipo. Vazio -> NaN."""
    if not isinstance(a, str) or not isinstance(b, str) or not a or not b:
        return np.nan
    return round(fuzz.token_sort_ratio(tirar_tipo(normalizar_nome(a)), tirar_tipo(normalizar_nome(b))), 0)

sug_d["score_nome"] = [score_nome(a, b) for a, b in zip(sug_d["logradouro"], sug_d["via_top"])]   # poucos milhares de pares
ceps_em_c = set(sug_c.loc[sug_c["status"].isin(["sugerido", "votacao_fraca"]), "cep_sugerido"])
nl = sug_d["n_locais"].fillna(0)
sug_d["status"] = np.select(
    [nl == 0, sug_d["n_locais_com_seg"].fillna(0) == 0, sug_d["melhor_status"] == "sugerido",
     sug_d["melhor_status"] == "votacao_fraca", sug_d["cep"].isin(ceps_em_c)],
    ["sem_pontos_cnefe", "pontos_longe", "sugerido", "votacao_fraca", "em_trecho_ambiguo"], default="sem_trecho_vencido")
sug_d = sug_d.merge(vinc, on="cep", how="left")
sug_d = sug_d.sort_values(["vinc_ativos_soma_anos", "n_locais"], ascending=False, na_position="last").reset_index(drop=True)
log.info("CEPs órfãos por status:\n%s", sug_d["status"].value_counts().to_string())
com_vinc = sug_d["vinc_ativos_soma_anos"].fillna(0) > 0
log.info("Órfãos com vínculos na rais: %d | destes com trecho sugerido: %d", int(com_vinc.sum()),
         int((com_vinc & sug_d["status"].isin(["sugerido", "em_trecho_ambiguo"])).sum()))
validar_tabela("Sugestões classe D (por CEP)", sug_d, chave=["cep"], obrigatorias=["cep", "status"], linhas_entrada=len(orfaos))
log.info("Classe D concluída!")

## 8. Sugestões por segmento (camada para o QGIS)
Um registro por segmento com sugestão: trechos ambíguos (parte B, `origem = B_trecho_ambiguo`) e segmentos votados por CEPs órfãos (parte C, `origem = C_cep_orfao`).
- `acao = preencher`: o segmento está **sem** CEP → copie `cep_sugerido` para `cep`;
- `acao = conferir`: o segmento **já tem** CEP e os pontos do CNEFE indicam outro (pode ser rua com mais de um CEP, ou CEP errado).

In [ ]:
log.info("Montando as sugestões por segmento...")
rank = ["sugerido", "votacao_fraca"]
s_c = sug_c[sug_c["status"].isin(rank)].assign(origem="B_trecho_ambiguo")[
    ["id", "cep_sugerido", "origem", "status", "participacao", "n_locais_vencedor", "n_votos_validos", "votos"]]
s_d = seg_d.rename(columns={"id_seg": "id"}).assign(origem="C_cep_orfao")[
    ["id", "cep_sugerido", "origem", "status", "participacao", "n_locais_vencedor", "n_votos_validos", "votos"]]
s_d["id"] = s_d["id"].astype(base["id"].dtype)
todas = pd.concat([s_c, s_d], ignore_index=True).rename(columns={"status": "status_sugestao"})
sugestoes = (base[["id", "tipo", "nome", "loteamento", "status_cep", "cep", "cep_origem", "geometry"]]
             .rename(columns={"cep": "cep_atual"}).merge(todas, on="id", how="inner"))
sugestoes["acao"] = np.where(sugestoes["cep_atual"] == "", "preencher", "conferir")
sugestoes = sugestoes.merge(vinc[["cep", "vinc_ativos_soma_anos"]].rename(columns={"cep": "cep_sugerido"}), on="cep_sugerido", how="left")
log.info("Segmentos com sugestão: %d | sugerido: %d | votação fraca: %d | preencher: %d | conferir: %d", len(sugestoes),
         int((sugestoes["status_sugestao"] == "sugerido").sum()), int((sugestoes["status_sugestao"] == "votacao_fraca").sum()),
         int((sugestoes["acao"] == "preencher").sum()), int((sugestoes["acao"] == "conferir").sum()))

# --- Camada de auditoria: segmentos dos CEPs suspeitos/atenção --------------------------------------------------------------
aud_seg = base[["id", "via", "loteamento", "cep_origem", "geometry"]].merge(
    pares.loc[pares["cep"].isin(alerta), ["id", "cep"]], on="id").merge(
    aud[["cep", "status", "n_locais", "dist_mediana_m", "pct_perto", "via_dos_pontos", "ids_dos_pontos"]], on="cep")

# --- Pontos usados (para ver no mapa) -------------------------------------------------------------------------------------------
uso = pd.Series("", index=locais.index)
uso[locais["cep"].isin(alerta)] = "A_auditoria"
uso[locais["cep"].isin(set_orf)] = "C_cep_orfao"
uso[locais["loc_id"].isin(votos["loc_id"])] = "B_trecho_ambiguo"
pontos = locais[uso != ""].assign(uso=uso[uso != ""])[["loc_id", "cep", "n_enderecos", "rua_cnefe", "id_seg", "dist_seg", "uso", "geometry"]]
pontos["dist_seg"] = pontos["dist_seg"].round(1)
log.info("Pontos do cnefe na camada de apoio: %d", len(pontos))
validar_tabela("Sugestões por segmento", sugestoes, chave=["id"], obrigatorias=["id", "cep_sugerido"])
log.info("Sugestões por segmento prontas!")

## 9. Exportação
Arquivos **regeráveis** (substituídos a cada execução): `auditoria_cnefe.gpkg` e `auditoria_cnefe.xlsx`.

No QGIS: abra `sugestoes_segmentos` junto com a base, filtre por `status_sugestao = 'sugerido'` e `acao = 'preencher'`, confira e copie `cep_sugerido` para `cep` da base (o `id` é o mesmo). A aba **Segmentos sugeridos** do Excel tem a mesma lista.

In [ ]:
log.info("Gravando o GeoPackage e o Excel...")
if ARQ_GPKG.exists():
    ARQ_GPKG.unlink()                                                     # recomeça do zero (arquivo regerável)
camadas = [("sugestoes_segmentos", sugestoes), ("auditoria_segmentos", aud_seg), ("pontos_cnefe", pontos)]
for i, (nome, gdf) in enumerate(camadas):
    if len(gdf) == 0:
        log.warning("Camada %s vazia: não gravada.", nome)
        continue
    para_gpkg(gdf).to_file(ARQ_GPKG, layer=nome, driver="GPKG", engine="pyogrio", **({"mode": "a"} if ARQ_GPKG.exists() else {}))
    log.info("  camada %s: %d feições", nome, len(gdf))

resumo = pd.DataFrame([
    {"parte": "A - auditoria", "status": k, "quantidade": int(v)} for k, v in aud["status"].value_counts().items()] + [
    {"parte": "B - classe C (trechos)", "status": k, "quantidade": int(v)} for k, v in sug_c["status"].value_counts().items()] + [
    {"parte": "C - classe D (CEPs órfãos)", "status": k, "quantidade": int(v)} for k, v in sug_d["status"].value_counts().items()])
col_c = ["id", "via", "loteamento", "ceps_candidatos", "status", "cep_sugerido", "participacao", "n_locais_vencedor",
         "n_votos_validos", "votos", "n_locais_outros_ceps", "vinc_ativos_soma_anos"]
col_d = ["cep", "logradouro", "bairro", "status", "ids_sugeridos", "n_segmentos_sugeridos", "via_top", "score_nome",
         "participacao", "n_locais_top", "n_locais_com_seg", "n_locais", "n_enderecos", "rua_cnefe", "classe",
         "vinc_ativos_ref", "vinc_ativos_soma_anos"]
col_s = ["id", "tipo", "nome", "loteamento", "status_cep", "cep_atual", "cep_origem", "acao", "cep_sugerido", "status_sugestao",
         "origem", "participacao", "n_locais_vencedor", "n_votos_validos", "votos", "vinc_ativos_soma_anos"]
with pd.ExcelWriter(ARQ_EXCEL, engine="openpyxl") as escritor:
    for nome_aba, tabela in [("Resumo", resumo), ("A - Auditoria CEPs", aud), ("B - Trechos ambiguos (C)", sug_c[col_c]),
                             ("C - CEPs orfaos (D)", sug_d[col_d]),
                             ("Segmentos sugeridos", pd.DataFrame(sugestoes)[col_s])]:
        tabela.to_excel(escritor, sheet_name=nome_aba, index=False)
        formatar_excel(escritor, nome_aba)
log.info("Excel salvo: %s", ARQ_EXCEL.name)
log.info("Exportação concluída!")

## 10. Resumo

In [ ]:
log.info("Resumo final...")
log.info("A - CEPs da base auditados: %d | suspeitos: %d | atenção: %d | ok: %d | sem pontos no cnefe: %d", len(aud),
         int((aud["status"] == "suspeito").sum()), int((aud["status"] == "atencao").sum()), int((aud["status"] == "ok").sum()),
         int((aud["status"] == "sem_pontos_cnefe").sum()))
log.info("B - trechos ambíguos: %d | com CEP sugerido: %d | votação fraca: %d | sem pontos: %d", len(sug_c),
         int((sug_c["status"] == "sugerido").sum()), int((sug_c["status"] == "votacao_fraca").sum()), int((sug_c["status"] == "sem_pontos").sum()))
log.info("C - CEPs órfãos: %d | com trecho sugerido: %d | em trecho ambíguo: %d | votação fraca: %d | sem pontos no cnefe: %d", len(sug_d),
         int((sug_d["status"] == "sugerido").sum()), int((sug_d["status"] == "em_trecho_ambiguo").sum()),
         int((sug_d["status"] == "votacao_fraca").sum()), int((sug_d["status"] == "sem_pontos_cnefe").sum()))
log.info("Segmentos para revisar no QGIS: %d (preencher: %d | conferir: %d)", len(sugestoes),
         int((sugestoes["acao"] == "preencher").sum()), int((sugestoes["acao"] == "conferir").sum()))
v_d = sug_d.loc[sug_d["status"].isin(["sugerido", "em_trecho_ambiguo"]), "vinc_ativos_soma_anos"].fillna(0).sum()
log.info("Vínculos (soma dos anos) em CEPs órfãos com trecho sugerido: %d", int(v_d))
top = aud[aud["status"] == "suspeito"].head(10)
if len(top):
    log.info("Maiores suspeitos (por vínculos):\n%s", top[["cep", "vias_base", "dist_mediana_m", "n_locais", "via_dos_pontos", "vinc_ativos_soma_anos"]].to_string(index=False))
log.info("Arquivos: %s e %s (pasta %s)", ARQ_GPKG.name, ARQ_EXCEL.name, PASTA_SAIDA)
log.info("Próximo: revisar no QGIS, copiar os CEPs aceitos para a base e rodar python executar_tudo.py --ciclo.")
log.info("Etapa 07 concluída!")